In [10]:
# %matplotlib widget
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from helper_functions import filtering, analysis, plotting, diagnostics
from obspy import read
import traceback 
import time
import obspy

STATION_PHASE_MAP = {
    "ARMA": "sks_deep", 
    "AS31": "sks", 
    "BBOO": "sks_deep",
    "BKB": "all",
    "BKNI": "complex",
    "BLDU": "sks",
    "CAN": "sks_deep",
    "CMSA": "complex",
    "CNB": "sks_deep",
    "COEN": "sks",
    "CTA": "complex",
    "CTAO": "complex",
    "EIDS": "sks_deep",
    "FAKI": "complex",
    "FITZ": "sks_deep",
    "FORT": "all",
    "GIRL": "all",
    "GUMO": "complex",
    "HTT": "complex",
    "INKA": "complex",
    "JAGI": "complex",
    "KDU": "complex",
    "KMBL": "complex",
    "KNRA": "complex",
    "LCRK": "complex",
    "LHI": "complex",
    "LUWI": "complex",
    "MANU": "complex",
    "MEEK": "complex",
    "MMRI": "sks_deep",
    "MNAI": "all",
    "MOO": "complex",
    "MORW": "complex",
    "MTN": "sks_deep",
    "MULG": "complex",
    "MUN": "sks_deep",
    "NFK": "complex",
    "NOUC": "sks_deep",
    "NWAO": "complex",
    "OOD": "complex",
    "PMBI": "complex", #potentially a null station?
    "PMG": "complex",
    "QIS": "sks",
    "QLP": "complex",
    "RABL": "complex",
    "RIV": "complex", # another potential null
    "SAUI": "complex",
    "SMRI": "complex",
    "SNZO": "all",
    "SOEI": "complex",
    "STKA": "sks_deep",
    "TNTI": "sks_deep",
    "TOO": "sks_deep",
    "UGM": "complex",
    "WB2": "complex",   # good null
    "WRKA": "complex", # super good null
    "XMI": "all",
    "XMIS": "all",
    "YNG": "sks_deep",
    }


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# ==================== Load data ====================


## the next two datasets are older and have the even 300 300 phase windows, along with older sta lta values

# dataType = 'sample'
# networkName = 'CAN'
# stationName = 'CAN'
# freqType = 'normal'

# raw_eventdata = pd.read_csv(
#     f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_03102026.txt',
#     sep='|'
# )

# dataType = 'complete'
# networkName = 'AU'
# stationName = 'australia'
# freqType = 'normal'

# raw_eventdata = pd.read_csv(
#     f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_03162026.txt',
#     sep='|'
# )

# the next two datasets are 200 400 phase windowed datasets with newer updated sta lta values

dataType = 'complete'
networkName = 'swp'
stationName = 'swp'
freqType = 'normal'

raw_swp_eventdata = pd.read_csv(
    f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_04032026.txt',
    sep='|'
)

dataType = 'complete'
networkName = 'AU'
stationName = 'australia'
freqType = 'normal'

raw_aus_eventdata = pd.read_csv(
    f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_04032026.txt',
    sep='|'
)

raw_eventdata = pd.concat([raw_swp_eventdata, raw_aus_eventdata], ignore_index=True)
raw_eventdata = raw_eventdata.sort_values('evstation')

In [12]:
# ==================== Prepare data ====================
# Calculate SI errors
eventData = filtering.calculate_si_errors(raw_eventdata)

print(f"Events before filtering: {len(eventData)}")

# count = (eventData["evphase"] == 'S').sum()
# print(count)

epicentral_max=360
epicentral_min=0
depth_min=0
depth_max=1e5
snrbefore_min=3.0
snrafter_min=1.0
sta_lta_min=0
mag_max=7.0
ellipticity_min=1.0
ellipticity_max=1e4
si_error_max=0.5
incident_max=90
pierce_depth_min=0
delta_travel_max = 25


# Apply quality filters
eventData = filtering.filter_events(
    eventData,
    epicentral_max=epicentral_max,
    epicentral_min=epicentral_min,
    depth_min=depth_min,
    depth_max=depth_max,
    snrbefore_min=snrbefore_min,
    snrafter_min=snrafter_min,
    sta_lta_min=sta_lta_min,
    mag_max=mag_max,
    ellipticity_min=ellipticity_min,
    ellipticity_max=ellipticity_max,
    si_error_max=si_error_max,
    incident_max=incident_max,
    pierce_depth_min=pierce_depth_min,
    delta_travel_max=delta_travel_max
)

eventData = eventData.drop_duplicates(subset=['evtime','evstation'])

# eventData = eventData[(eventData['evphase']=='SKS')]
# eventData = eventData[(eventData['wiggleSI_calc']>=0.5)]

# Remove shallow events with large incident angles (keeps only events with large epicentral distances and smaller incident angles)
eventData = eventData[~((eventData['evdp'] < 400) & (eventData['evincident'] > 25))]

# # Remove deep events with very large incident angles (sample)
eventData = eventData[~((eventData['evdp'] >= 400) & (eventData['evincident'] > 35))]

# ^^^^^^ these two filters are heavily used in the new analysis so keep them!!!

# eventData = eventData[eventData['evdp'] >= 400]


'''
april 5 notes
- added tilt to SI calculation and plotting (see analysis.py and spring_updates.ipynb)
- started looking at treating deep S phases and direct S phases separately (mainly dealing with incident angle and pierce depth filters)
- analysis of only deep S phases to identify trends and how this could be corraborated with sampling of anisotropy at tilted layer)
'''

print(f"Events after filtering: {len(eventData)}")

count = (eventData["evphase"] == 'S').sum()
print(count)

Events before filtering: 99224
Events after filtering: 8012
7089


In [5]:
eventData.to_csv('/work/gcl3/BR/nicolasv03/senior_thesis/measurements/swp_event_data/eventData.txt',sep='\t')

# Older Functions: Plot Grid of Selected Waveforms

In [ ]:
# Define grid layout
n_plots = 42
ncols = 3
nrows = int(np.ceil(n_plots / ncols))

truncatedData = eventData.copy()
truncatedData = truncatedData[truncatedData['evstation']=='ARMA']
truncatedData = truncatedData[truncatedData['evphase']=='SKS']
# truncatedData = truncatedData[truncatedData['evdp']>=400]
truncatedData = truncatedData[(truncatedData['calcbaz']>=120) & (truncatedData['calcbaz']<=150)]
# truncatedData = truncatedData[truncatedData['wiggleSI_calc']>=0]
truncatedData = truncatedData.sort_values(by='wiggleSI_calc')



eventData_shuffled = truncatedData
# eventData_shuffled = truncatedData.sample(frac=1).reset_index(drop=True)

# Create figure with subplots that have 2 columns each (waveform + particle motion)
fig = plt.figure(figsize=(ncols*8, nrows*3))
gs = fig.add_gridspec(nrows, ncols*2, hspace=0.3, wspace=0.3)

for i in range(n_plots):
    split = eventData_shuffled.iloc[i]
    print(f"Processing {i+1}/{n_plots}: {split['waveform_path']}")
    
    row = i // ncols
    col = (i % ncols) * 2  # Each station gets 2 columns
    
    try:
        wf = read(split['waveform_path'])
        wf.detrend('demean')
        wf.detrend('linear')
        wf.filter("bandpass", freqmin=0.01, freqmax=0.125)

        north = wf.select(component="N", channel='BH*')[0].data
        east = wf.select(component="E", channel='BH*')[0].data
        sample_interval = wf[0].stats.delta
        time = np.arange(len(north)) * sample_interval

        # Window indices
        windowstart = int(split['windowstart'] / sample_interval)
        windowend = int(split['windowend'] / sample_interval)
        
        # ========== Left: Waveform ==========
        ax_wf = fig.add_subplot(gs[row, col])
        ax_wf.plot(time, north, c='b', label='North', linewidth=0.5)
        ax_wf.plot(time, east, c='r', label='East', linewidth=0.5)
        ax_wf.axvline(split['windowstart'], color='black', linestyle='--', linewidth=1.0, alpha=0.8)
        ax_wf.axvline(split['windowend'], color='black', linestyle='--', linewidth=1.0, alpha=0.8)
        ax_wf.set_title(f"{split['evstation']} - {split['evphase']} , {round(split['ellipticity'],2)} SI: {round(split['wiggleSI_calc'],2)}", fontsize=12)
        ax_wf.tick_params(labelsize=6)
        
        if i == 0:
            ax_wf.legend(fontsize=6)
        
        # ========== Right: Particle Motion ==========
        ax_pm = fig.add_subplot(gs[row, col + 1])
        
        # Use windowed data for particle motion
        north_window = north[windowstart:windowend]
        east_window = east[windowstart:windowend]
        
        # Create pair for particle motion
        import splitwavepy as sw
        if len(north_window) % 2 == 0:
            north_window = north_window[:-1]
            east_window = east_window[:-1]
        
        pair = sw.Pair(north_window, east_window, delta=sample_interval)
        
        # Use your trace_ppm function from support
        from support import trace_ppm
        trace_ppm(pair, ax_pm, 0, len(north_window), 
                 labels=['N', 'E'], baz=f"{split['evbaz']:.0f}°")
        
    except Exception as e:
        print(f"Failed on {i}: {e}")
        ax_wf = fig.add_subplot(gs[row, col])
        ax_wf.text(0.5, 0.5, 'Failed', ha='center', va='center')
        ax_wf.set_title(f"Error: {i}", fontsize=8)
        ax_wf.axis('off')
        
        ax_pm = fig.add_subplot(gs[row, col + 1])
        ax_pm.axis('off')

fig.suptitle(f"Sample of Waveforms, snrbefore={snrbefore_min}, snrafter={snrafter_min}, ellipmin={ellipticity_min}, SIerrmax={si_error_max}, epicmin={epicentral_min}", 
             fontsize=16, y=0.995)

plt.tight_layout()
plt.show()

# eventData Histograms

In [ ]:
# eventData = truncatedData

plt.figure(figsize=(12,6))
plt.hist(eventData['evmg'], bins=np.linspace(5,8,11), alpha=0.5, label='Event Magnitude')
plt.xlim(5,8)
plt.xticks(np.linspace(5,8,11))
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.title('Distribution of Event Magnitudes')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
plt.hist(eventData['snrbefore'], bins=np.linspace(0,20,201), alpha=0.5, label='SNR Before')
plt.xlim(0,20)
plt.xticks(np.linspace(0,20,21))
plt.xlabel('SNR')
plt.ylabel('Frequency')
plt.title('Distribution of SNR Before')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
plt.hist(eventData['snrafter'], bins=np.linspace(0,10,201), alpha=0.5, label='SNR After')
plt.xlim(0,10)
plt.xticks(np.linspace(0,10,21))
plt.xlabel('SNR')
plt.ylabel('Frequency')
plt.title('Distribution of SNR After')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
si_errors = eventData['SI_calc_error']
plt.hist(si_errors, bins=np.linspace(0,4,101), alpha=0.5, label='SI')

# plt.xlim(0,20)
# plt.xticks(np.linspace(0,20,21))
plt.xlabel('SI error')
plt.ylabel('Frequency')
plt.title('Distribution of SI error')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
si_errors = eventData['ellipticity']
plt.hist(si_errors, alpha=0.5, label='Ellipticity')
# plt.xlim(0,20)
# plt.xticks(np.linspace(0,20,21))
plt.xlabel('Ellipticity')
plt.ylabel('Frequency')
plt.title('Distribution of Ellipticity')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
si_errors = eventData['sta_lta']
plt.hist(si_errors, bins=20, alpha=0.5, label='STA/LTA')
# plt.xlim(0,20)
# plt.xticks(np.linspace(0,20,21))
plt.xlabel('STA/LTA')
plt.ylabel('Frequency')
plt.title('Distribution of STA/LTA')
plt.legend()
plt.show()



# Generate Plots for Single Defined Station

In [ ]:
print(eventData['evstation'].unique())

truncatedData = eventData.copy()
truncatedData = truncatedData[truncatedData['evstation']=='EIDS']
truncatedData = truncatedData[truncatedData['evphase']=='SKS']
# # truncatedData = truncatedData[truncatedData['evdp']>=200]
truncatedData = truncatedData[(truncatedData['evbaz']>=135) & (truncatedData['evbaz']<=150)]
# # truncatedData = truncatedData[truncatedData['wiggleSI_calc']>=0]


truncatedData = truncatedData[['evstation','evtime','evlat','evlon','evdp','evmg','evphase','evepic','evbaz','calcbaz','wiggleSI_calc']]
truncatedData = truncatedData.sort_values(by='evepic')

truncatedData


s_truncatedData = truncatedData[truncatedData['evphase']=='S']
sks_truncatedData = truncatedData[truncatedData['evphase']=='SKS']

sks_shallow_truncatedData = sks_truncatedData[sks_truncatedData['evdp']<400]
sks_deep_truncatedData = sks_truncatedData[sks_truncatedData['evdp']>=400]

# plt.plot(truncatedData['evbaz'], truncatedData['wiggleSI_calc'], 'o',c='b')
# plt.plot(truncatedData['calcbaz'], truncatedData['wiggleSI_calc'], 'o',c='r')

# plt.plot(s_truncatedData['evbaz'], s_truncatedData['wiggleSI_calc'], 'o',c='r', label='S')
# plt.plot(sks_truncatedData['evbaz'], sks_truncatedData['wiggleSI_calc'], 'o',c='b', label='SKS')

plt.figure(figsize=(8,6))
# plt.plot(s_truncatedData['evepic'], s_truncatedData['wiggleSI_calc'], 'o',c='r', label='S')
plt.plot(sks_truncatedData['evepic'], sks_truncatedData['wiggleSI_calc'], 'o',c='blue', label='SKS')

# plt.plot(sks_shallow_truncatedData['evepic'], sks_shallow_truncatedData['wiggleSI_calc'], 'o',c='blue', label='SKS shallow')
# plt.plot(sks_deep_truncatedData['evepic'], sks_deep_truncatedData['wiggleSI_calc'], 'o',c='green', label='SKS deep')
plt.xlabel('Epicentral Distance [deg]')
# plt.xlim(100,200)
plt.ylabel('SI')
# plt.yticks(np.linspace(-2,2,9))
# plt.ylim(-2,2)
plt.legend()
# plt.title(' SKS events with 120-150 baz')
plt.show()

print(f" Mean source latitude: {round(np.mean(truncatedData['evlat']),2)}")
print(f" Mean source longitude: {round(np.mean(truncatedData['evlon']),2)}")


truncatedData.to_csv('/work/gcl3/BR/nicolasv03/senior_thesis/codes/Plotting/EIDS_evbaz_135_to_150_events.txt')

In [ ]:


# Select station
# station_events = eventData 
station_code = 'BCON'
station_events = eventData[eventData['evstation'] == station_code]
# station_events = station_events[station_events['evphase'] == 'SKS'] 
# station_events = station_events[~(station_events['calcbaz']>=300) & (station_events['calcbaz']<=350)]
# station_events = station_events[(station_events['calcbaz']<120)|(station_events['calcbaz']>150)]
# station_events = station_events[(~station_events['calcbaz']<120)|(~station_events['calcbaz']>150)]

# # SKS and deep S only filter:
# station_events = station_events[
#     (station_events['evphase'] == 'SKS') |  # Keep if SKS
#     ((station_events['evphase'] == 'S') & (station_events['evdp'] >= 400)) # OR if deep S
# ]
    

# ARMA SKS cluster removal: between 120 - 150 
# BBOO SKS cluster removal: between 145 - 165
# station_events = station_events[
#     (station_events['evphase'] != 'SKS') |  # Keep if not SKS
#     (station_events['calcbaz'] < 120) |      # OR if calcbaz < 145
#     (station_events['calcbaz'] > 150)      # OR if calcbaz > 165
# ]

# station_events = station_events[
#     (station_events['evphase'] != 'SKS') |  # Keep if not SKS
#     (station_events['calcbaz'] < 115) |      # OR if calcbaz < 145
#     (station_events['calcbaz'] > 125)      # OR if calcbaz > 165
# ]

# # # BBOO remove S above 200 baz 
# station_events = station_events[
#     (station_events['evphase'] != 'S') |  # Keep if not S
#     (station_events['calcbaz'] < 200)      # OR if calcbaz < 200
# ]

# EIDS remove super strong deep S SI values above 1.5
# MULG  remove super strong deep S SI values above 1.8
# SNZO  remove super strong deep S SI values above 2
# TOO  remove super strong deep S SI values above 1.4



# station_events = station_events[abs(station_events['wiggleSI_calc'])<1.8]

bin_width = 20
bin_minimum = 3
output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/spring_plots/{station_code}/'

# # ========== Plot 1: SI Curves for Different Phase Types ==========
results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False,with_C=True)
# fig1, stats = analysis.plot_station_comparison(results, station_code, s_depth=400)
# # # plt.savefig(output_path+f'AllSIPlots_bin_width_{bin_width}_min_{bin_minimum}.png', dpi=300, bbox_inches='tight')
# plt.show()


# # ========== Plot 2: S Phase Depths and Back-Azimuths ==========
# fig2, axes = plotting.plot_s_phases_dual_view(station_events, station_code, s_depth=400)
# # plt.savefig(output_path+f'_sPhase_depth_and_back_azimuth.png', dpi=300, bbox_inches='tight')
# plt.show()

# # Plot 3: Comprehensive 6-panel comparison
# fig3, axes = plotting.plot_s_phase_parameter_comparison(station_events, station_code)
# plt.show()

# # Plot 4: Incidence and pierce depth focused analysis
# fig4, axes = plotting.plot_incidence_pierce_depth_grid(station_events, station_code)
# plt.show()

# # decide which of these two to use from previous plots, then uncomment to save figure

# fig1, stats = analysis.plot_station_sks_deep(results, station_code, s_depth=400,figsize=(12,8))
# # plt.savefig(output_path+f'bin_width_{bin_width}_min_{bin_minimum}_sks_only.png', dpi=300, bbox_inches='tight')
# plt.show()

# fig5, stats = analysis.plot_station_s_only(results, station_code, s_depth=400,figsize=(12,8))
# # plt.savefig(output_path+f'bin_width_{bin_width}_min_{bin_minimum}.png', dpi=300, bbox_inches='tight')
# plt.show()

# fig7, stats_all = analysis.plot_all_phases(results, station_code, s_depth=400, figsize=(12, 8))
# # plt.savefig(output_path+f'bin_width_{bin_width}_min_{bin_minimum}.png', dpi=300, bbox_inches='tight')
# plt.show()
# fig6, ax6 = plotting.plot_station_stereographic_map(results, station_code,STATION_PHASE_MAP[station_code], s_depth=400, figsize=(10, 10))
# plt.savefig(output_path+f'{station_code}_event_distribution_map.png', dpi=300, bbox_inches='tight')
# plt.show()

fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=True)
# plt.savefig(output_path+f'{station_code}_draft_publication_plot_with_other_curve.png', dpi=300, bbox_inches='tight')
plt.show()


# # ========== Station Summary ==========
# print("\n" + "="*70)
# print(f"STATION SUMMARY: {station_code}")
# print("="*70)

# # Location
# print(f"\nLocation:")
# print(f"  Latitude:  {station_events['evstationlat'].iloc[0]:.3f}°")
# print(f"  Longitude: {station_events['evstationlon'].iloc[0]:.3f}°")
# print(f"  Network:   {station_events['evnetwork'].iloc[0] if 'evnetwork' in station_events.columns else 'N/A'}")

# # Date range
# if 'evtime' in station_events.columns:
#     # Extract years from evtime (assumes format like 'YYYY-MM-DD...')
#     dates = pd.to_datetime(station_events['evtime'], errors='coerce')
#     print(f"  Date range: {dates.min().year} - {dates.max().year}")

# # Measurements by phase type
# n_sks = len(station_events[station_events['evphase'] == 'SKS'])
# n_s_deep = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] >= 400)])
# n_s_shallow = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] < 400)])

# print(f"\nMeasurements:")
# print(f"  SKS:              {n_sks}")
# print(f"  S deep (≥400km):  {n_s_deep}")
# print(f"  S shallow:        {n_s_shallow}")
# print(f"  Total:            {len(station_events)}")

# # Magnitude range
# print(f"\nMagnitude range: Mw {station_events['evmg'].min():.1f} - {station_events['evmg'].max():.1f}")

# # Splitting parameters (from different phase combinations in stats)
# print(f"\nSplitting Parameters:")
# for phase_key, label in [('sks', 'SKS only'), ('sks_deep', 'SKS + Deep S'), ('all', 'All phases')]:
#     if phase_key in stats and stats[phase_key]:
#         s = stats[phase_key]
#         param_str = f"  {label:15s}: φ={s['phi_binned']:6.1f}°, δt={s['dt_binned']:.3f}s"
#         if 'tilt_binned' in s and s['tilt_binned'] is not None:
#             param_str += f", tilt={s['tilt_binned']:.1f}°"
#         print(param_str)

# print("="*70 + "\n")

In [ ]:
print(results['sks_deep']['result_binned']['cov'])
pcov = results['sks_deep']['result_binned']['cov']
perr = np.sqrt(np.diag(pcov))
print(perr)

In [ ]:
results['sks_deep']['result_binned']

# Loop Function: Generate Plots For ALL Stations in eventData

In [ ]:
# Parameters
bin_width = 30
bin_minimum = 3
s_depth = 400

# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 10].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Loop through stations
for station_code in station_list:
    print(f"\n{'='*70}")
    print(f"Processing {station_code}...")
    print('='*70)
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]
        output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/spring_plots/april_10/{station_code}'
        
        # ========== Plot 1: SI Curves for Different Phase Types ==========
        print(f"  Generating SI curve plot...")
        results = analysis.analyze_station_comparison(
            eventData, 
            station_code, 
            bin_width=bin_width, 
            bin_minimum=bin_minimum, 
            s_depth=s_depth
        )
        
        fig1, stats = analysis.plot_station_comparison(results, station_code, s_depth=s_depth)
        plt.savefig(output_path + f'_si_curve_width_{bin_width}_min_{bin_minimum}.png', 
                   dpi=300, bbox_inches='tight')
        plt.close()  # Close instead of show to save memory
        print(f"    ✓ Saved SI curve plot")
        
        # ========== Plot 2: S Phase Depths and Back-Azimuths ==========
        print(f"  Generating S-phase depth/baz plot...")
        fig2, axes = plotting.plot_s_phases_dual_view(station_events, station_code, s_depth=s_depth)
        
        if fig2 is not None:  # Check if S-phase data exists
            plt.savefig(output_path + f'_sPhase_depth_and_back_azimuth.png', 
                       dpi=300, bbox_inches='tight')
            plt.close()
            print(f"    ✓ Saved S-phase plot")
        else:
            print(f"    - No S-phase data for this station")
        
        print(f"  ✓ Completed {station_code}")
        
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        plt.close('all')  # Clean up any open figures
        continue

print(f"\n{'='*70}")
print(f"✓ Processing complete for {len(station_list)} stations")
print('='*70)

# new bootstrapping method

In [ ]:
# SKS + Deep S
eventData_sks_deep = eventData[(eventData['evphase'] == 'SKS') | 
                                ((eventData['evphase'] == 'S') & (eventData['evdp'] >= 400))]

eventData_sks_deep = eventData_sks_deep[(eventData_sks_deep['calcbaz']<120)|(eventData_sks_deep['calcbaz']>150)]

# eventData_sks_deep = eventData

station_code = 'ARMA'
# Run bootstrap (no tilt)
bootstrap_result = analysis.bootstrap_splitting_parameters(
    eventData_sks_deep,
    station_code=station_code,
    bin_width=20,
    bin_minimum=3,
    n_bootstrap=100,
    use_tilt=False,
    subsample_fraction=0.5
)

# Plot results
fig, axes = plotting.plot_bootstrap_results(
    bootstrap_result,
    bootstrap_result['binned'],
    station_code=station_code
)
plt.show()

# # Or with tilt
# bootstrap_result_tilt = analysis.bootstrap_splitting_parameters(
#     eventData,
#     station_code='ARMA',
#     n_bootstrap=1000,
#     use_tilt=True
# )

# fig, axes = plotting.plot_bootstrap_results(bootstrap_result_tilt, bootstrap_result_tilt['binned'], 'ARMA')
# plt.show()

# loop to get splitting parameters

In [4]:
'''
Doing this just to get splitting parameters for the stations so we can start making a publiation quality map. Final measurements coming soon!! May 11 2026

'''

# Parameters
bin_width = 20
bin_minimum = 3
s_depth = 400

# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 1].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Filter to only stations in dictionary
station_list = [s for s in station_list if s in STATION_PHASE_MAP]
print(f"After dict filter: {len(station_list)} stations remaining")

print(station_list)

plotting_parameters = pd.DataFrame(columns=["Station","Network","Latitude","Longitude","phi","dt","sks_count","s_deep_count","s_shallow_count","total_count"])

# Loop through stations
for station_code in station_list:
    time.sleep(1)
    print(f"\n{'='*70}")
    print(f"Processing {station_code}...")
    print('='*70)
    time.sleep(1)
    # if STATION_PHASE_MAP[station_code] == 'complex':
    #     print("Station is complex and will not be analyzed.")
    #     continue
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]

        # print(station_events.head(3))
        results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False, with_C=False)

        ####################

        # Use phase from station phase mapping
        data = results.get(STATION_PHASE_MAP[station_code])
        
        result_binned = data['result_binned']
        station_events = data['station_events']
        
        # Extract parameters
        phi_binned = round(result_binned['phi'],1)
        dt_binned = round(result_binned['dt'],2)

        station_lat = round(station_events['evstationlat'].iloc[0],2)
        station_lon = round(station_events['evstationlon'].iloc[0],2)

        network_name = station_events['evnetwork'].iloc[0]
        print(network_name)

        sks_count = len(station_events[station_events['evphase'] == 'SKS'])
        s_deep_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] >= s_depth)])
        s_shallow_count = len(station_events[(station_events['evphase'] == 'S') & (station_events['evdp'] < s_depth)])
        total_count = sks_count + s_deep_count + s_shallow_count
        print(sks_count,s_deep_count,s_shallow_count)




        # print(phi_binned,dt_binned,station_lat,station_lon)

        plotting_parameters.loc[len(plotting_parameters)] = [station_code,network_name,station_lat,station_lon,phi_binned,dt_binned,sks_count,s_deep_count,s_shallow_count,total_count]
        # ######################################################
        # print(data['station_events'].head())


        output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/test4_plots/'

        # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=True)
        fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=False)

        plt.savefig(output_path+f'{station_code}_{STATION_PHASE_MAP[station_code]}_plot.png', dpi=300, bbox_inches='tight')
        
        
        print(f"  ✓ Completed {station_code}")

        plt.close('all')

        results = []
        
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        plt.close('all')  # Clean up any open figures
        continue

# print(f"\n{'='*70}")
# print(f"✓ Processing complete for {len(station_list)} stations")
# print('='*70)

print(plotting_parameters)

plotting_parameters.to_csv("/work/gcl3/BR/nicolasv03/senior_thesis/measurements/publication_map/swp_measurements_test4.txt",sep='\t')

Found 83 unique stations
Stations: ['ARMA', 'AS31', 'BAG', 'BBOO', 'BCON', 'BKB', 'BKNI', 'BLDU', 'BTDF', 'CAN', 'CISI', 'CMSA', 'CNB', 'COEN', 'CTA', 'CTAO', 'DAV', 'DLV', 'EIDS', 'FAKI', 'FITZ', 'FORT', 'GIRL', 'GUMO', 'HNR', 'HTT', 'INKA', 'IPM', 'JAGI', 'JAY', 'KDU', 'KKM', 'KLGGI', 'KMBL', 'KNRA', 'KOM', 'KSM', 'KUM', 'LCRK', 'LDM', 'LHI', 'LUWI', 'MANU', 'MEEK', 'MILA', 'MMRI', 'MNAI', 'MOO', 'MORW', 'MTN', 'MULG', 'MUN', 'NFK', 'NOUC', 'NWAO', 'OOD', 'PATS', 'PBKT', 'PMBI', 'PMG', 'QIS', 'QLP', 'RABL', 'RIV', 'SAUI', 'SBM', 'SKLT', 'SMRI', 'SNZO', 'SOEI', 'STKA', 'SZP', 'TNTI', 'TOLI', 'TOO', 'TROP', 'UBIN', 'UGM', 'WB2', 'WRKA', 'XMI', 'XMIS', 'YNG']

Processing 83 stations with ≥10 measurements
After dict filter: 59 stations remaining
['ARMA', 'AS31', 'BBOO', 'BKB', 'BKNI', 'BLDU', 'CAN', 'CMSA', 'CNB', 'COEN', 'CTA', 'CTAO', 'EIDS', 'FAKI', 'FITZ', 'FORT', 'GIRL', 'GUMO', 'HTT', 'INKA', 'JAGI', 'KDU', 'KMBL', 'KNRA', 'LCRK', 'LHI', 'LUWI', 'MANU', 'MEEK', 'MMRI', 'MNAI', 'MOO

Traceback (most recent call last):
  File "/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py", line 1353, in analyze_station_comparison
    result_binned = analysis.analyze_station(
  File "/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py", line 865, in analyze_station
    fit_result = fit_splitting_intensity(
  File "/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py", line 533, in fit_splitting_intensity
    params = np.linalg.solve(XtWX, XtWy)
  File "/home/gcl/BR/nicolasv03/miniconda3/envs/jupyter_splitting_py310/lib/python3.10/site-packages/numpy/linalg/linalg.py", line 409, in solve
    r = gufunc(a, b, signature=signature, extobj=extobj)
  File "/home/gcl/BR/nicolasv03/miniconda3/envs/jupyter_splitting_py310/lib/python3.10/site-packages/numpy/linalg/linalg.py", line 112, in _raise_linalgerror_singular
    raise LinAlgError("Singular matrix")
numpy.linalg.LinAlgError: Singular matrix


  ✓ Completed BKB

Processing BKNI...
analyze_station_comparison: less than 3 events!
  ✗ Failed on BKNI: 'NoneType' object is not subscriptable

Processing BLDU...
AU
18 0 0
Fast Axis: 42.2613244619254, Delay Time: 0.36699412296478434
  ✓ Completed BLDU

Processing CAN...
G
27 41 0
Fast Axis: 15.407548203728838, Delay Time: 0.672282587883223
  ✓ Completed CAN

Processing CMSA...
  ✗ Failed on CMSA: 'NoneType' object is not subscriptable

Processing CNB...
AU
38 53 0
Fast Axis: 8.874284166654002, Delay Time: 0.42814797882127764
  ✓ Completed CNB

Processing COEN...
AU
22 0 0
Fast Axis: 155.85173338912324, Delay Time: 0.703198902135917
  ✓ Completed COEN

Processing CTA...
  ✗ Failed on CTA: 'NoneType' object is not subscriptable

Processing CTAO...
  ✗ Failed on CTAO: 'NoneType' object is not subscriptable

Processing EIDS...
AU
50 50 0
Fast Axis: 23.00094815826959, Delay Time: 0.4885580062039731
  ✓ Completed EIDS

Processing FAKI...
analyze_station_comparison: less than 3 events!
  ✗

/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py:568: RuntimeWarning: invalid value encountered in sqrt
  perr = np.sqrt(np.diag(pcov_physical))



Processing HTT...
Failed within analyze_station_comparison
Singular matrix
  ✗ Failed on HTT: 'NoneType' object is not subscriptable


...

  ✓ Completed SAUI

Processing SMRI...
analyze_station_comparison: less than 3 events!
Failed within analyze_station_comparison
Singular matrix
  ✗ Failed on SMRI: 'NoneType' object is not subscriptable


Traceback (most recent call last):
  File "/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py", line 1353, in analyze_station_comparison
    result_binned = analysis.analyze_station(
  File "/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py", line 865, in analyze_station
    fit_result = fit_splitting_intensity(
  File "/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py", line 533, in fit_splitting_intensity
    params = np.linalg.solve(XtWX, XtWy)
  File "/home/gcl/BR/nicolasv03/miniconda3/envs/jupyter_splitting_py310/lib/python3.10/site-packages/numpy/linalg/linalg.py", line 409, in solve
    r = gufunc(a, b, signature=signature, extobj=extobj)
  File "/home/gcl/BR/nicolasv03/miniconda3/envs/jupyter_splitting_py310/lib/python3.10/site-packages/numpy/linalg/linalg.py", line 112, in _raise_linalgerror_singular
    raise LinAlgError("Singular matrix")
numpy.linalg.LinAlgError: Singular matrix



Processing SNZO...
IU
16 5 22
Fast Axis: 38.60762185642952, Delay Time: 1.2208506410892273
  ✓ Completed SNZO

Processing SOEI...
analyze_station_comparison: less than 3 events!
analyze_station: binned measurements is 0
  ✗ Failed on SOEI: 'NoneType' object is not subscriptable

Processing STKA...
AU
44 87 0
Fast Axis: 17.233308168359223, Delay Time: 0.6773807192880488
  ✓ Completed STKA

Processing TNTI...
analyze_station_comparison: less than 3 events!
GE
0 29 0
Fast Axis: 9.90674761426713, Delay Time: 0.940209828026956
  ✓ Completed TNTI

Processing TOO...
AU
27 40 0
Fast Axis: 38.44405445778861, Delay Time: 0.6394099663853346
  ✓ Completed TOO

Processing UGM...
  ✗ Failed on UGM: 'NoneType' object is not subscriptable

Processing WB2...
  ✗ Failed on WB2: 'NoneType' object is not subscriptable

Processing WRKA...


/work/gcl3/BR/nicolasv03/senior_thesis/codes/helper_functions/analysis.py:568: RuntimeWarning: invalid value encountered in sqrt
  perr = np.sqrt(np.diag(pcov_physical))


  ✗ Failed on WRKA: 'NoneType' object is not subscriptable

Processing XMI...
AU
7 24 70
Fast Axis: 27.978299857925656, Delay Time: 0.5236526083375656
  ✓ Completed XMI

Processing XMIS...
analyze_station: binned measurements is 0
AU
5 22 77
Fast Axis: 32.18580773821821, Delay Time: 0.6187378512014342
  ✓ Completed XMIS

Processing YNG...
AU
34 39 0
Fast Axis: 176.24619570442485, Delay Time: 0.6536549182873964
  ✓ Completed YNG
   Station Network  Latitude  Longitude    phi    dt  sks_count  s_deep_count  \
0     ARMA      AU    -30.42     151.63   19.1  0.21         63            86   
1     AS31      AU    -23.67     133.90   71.8  0.76         15             0   
2     BBOO      AU    -32.81     136.06   69.0  0.50         40            83   
3      BKB      GE     -1.11     116.90   21.9  0.77          0             9   
4     BLDU      AU    -30.61     116.71   42.3  0.37         18             0   
5      CAN       G    -35.32     149.00   15.4  0.67         27            41   
6

# BLOB IDENTIFIER

In [ ]:
# Parameters
bin_width = 20
bin_minimum = 3
s_depth = 400

blob_stations = ['ARMA','BBOO','CAN','CNB','YNG']
# 'BCON'
# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 1].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Filter to only stations in dictionary and in blob list
station_list = [s for s in station_list if s in STATION_PHASE_MAP]
station_list = [s for s in station_list if s in blob_stations]

print(f"After dict filter: {len(station_list)} stations remaining")

model = obspy.taup.TauPyModel(model="iasp91")

from obspy.geodetics.base import locations2degrees,gps2dist_azimuth

from pyproj import Geod

g = Geod(ellps='WGS84')

blob_coords = pd.DataFrame(columns=['lat','lon','evlat','evlon','statlat','statlon','evdp','evphase','evepic','SI'])

blob_tick = 0 

# Loop through stations
for station_code in station_list:
    
    # if STATION_PHASE_MAP[station_code] == 'complex':
    #     continue
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]
        # print(station_code)
        # print(station_events.head(3))
        results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=False, with_C=False)

        ####################

        # Use phase from station phase mapping
        data = results.get(STATION_PHASE_MAP[station_code])
        # print(data)
        # print(data.keys())
        # time.sleep(1)
        events = data['station_events']
        blob_events = events[(events['evphase'] == 'SKS') & 
                             (events['calcbaz'] > 130) &
                             (events['calcbaz'] < 160) 
        ]

        # print(len(blob_events))
        for i in range(0,len(blob_events)):
            event = blob_events.iloc[i]
        
            arr = model.get_pierce_points(  source_depth_in_km=event['evdp'],
                                            distance_in_degree=event['evepic'],
                                            phase_list=['SKS'])
            
            arr2 = model.get_pierce_points_geo(source_depth_in_km=event['evdp'], source_latitude_in_deg=event['evlat'],
                              source_longitude_in_deg=event['evlon'],
                              receiver_latitude_in_deg=event['evstationlat'],
                              receiver_longitude_in_deg=event['evstationlon'],
                              phase_list=['SKS'],
                              )
            
            arrivals = model.add_geo_to_arrivals(arr2, source_latitude_in_deg=event['evlat'],
                              source_longitude_in_deg=event['evlon'],
                              receiver_latitude_in_deg=event['evstationlat'],
                              receiver_longitude_in_deg=event['evstationlon']
                                           )
            
            # pierce = np.array(arr[0].pierce)
            # cmb_inds = pierce[:]
            pierce_array = np.array(arr[0].pierce.tolist())
            cmb_inds = np.where(np.abs(pierce_array[:,3] - 2889) < 20)[0]
            receiver_cmb = pierce_array[cmb_inds[-1]]
            cmb_dist_deg = np.degrees(receiver_cmb[2])
            # print(receiver_cmb)
            # print(cmb_dist_deg)


            dist_m, az, baz = gps2dist_azimuth(event['evlat'],
                                                event['evlon'],
                                                event['evstationlat'],
                                                event['evstationlon'])


            total_deg = locations2degrees(
                                                event['evlat'],
                                                event['evlon'],
                                                event['evstationlat'],
                                                event['evstationlon'],
                                            )
            fraction = cmb_dist_deg / total_deg

            pierce_dist_m = dist_m * fraction

            # print(fraction)



            cmb_lon, cmb_lat, backaz = g.fwd(
                event['evlon'],
                event['evlat'],
                az,
                pierce_dist_m
            )


            # print(cmb_lat, cmb_lon)

            blob_coords.loc[blob_tick,:] = [cmb_lat,cmb_lon,event['evlat'],event['evlon'],event['evstationlat'],event['evstationlon'],event['evdp'],event['evphase'],event['evepic'],event['wiggleSI_calc']]

            blob_tick += 1

            

        # plot the blob
        # output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/blob_plots/'
        # fig7, ax7 = analysis.publication_SI_curve(results, station_code,STATION_PHASE_MAP[station_code],s_depth=400,figsize=(12,8),with_C=False)
        # plt.savefig(output_path+f'{station_code}_{STATION_PHASE_MAP[station_code]}_plot.png', dpi=300, bbox_inches='tight')
        
        # station_events = station_events[
#     (station_events['evphase'] != 'SKS') |  # Keep if not SKS
#     (station_events['calcbaz'] < 120) |      # OR if calcbaz < 145
#     (station_events['calcbaz'] > 150)      # OR if calcbaz > 165
# ]


        results = []
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        traceback.print_exc()

        plt.close('all')  # Clean up any open figures
        continue





In [ ]:
# blob_coords.to_csv

blob_coords.to_csv("/work/gcl3/BR/nicolasv03/senior_thesis/measurements/blob_plots/blob_coordinates.txt",sep='\t')

In [ ]:
data['station_events']

In [ ]:
print(np.mean(blob_coords['statlat']))
print(np.mean(blob_coords['statlon']))

In [ ]:
# blob_coords.sort_values()

blob_coords.head()

In [ ]:
arr2[0].pierce

vars(arr2[0])

print(arr2[0].pierce.dtype)